# 01 - EDA baseline: audit of the modelling dataset

**Role:** ZM - EDA and statistical analysis
**Deliverable:** `notebooks/01_eda.ipynb`, step 1 of the EDA work
**Scope of this step:** load the dataset through the official team chain and audit it.

This notebook performs **no imputation, no encoding, no scaling, no feature
selection and no statistical test**. It only describes the data as it is, so that
every later analysis starts from a frozen, verifiable baseline.

## Provenance note

- **Dataset version:** the file currently available at commit **`bd8a0cb`**
  (`data: add filtered dataset without ISR in data/raw`).
- **File used:** `data/raw/startup_data.csv`, 14 columns, Kaggle *Big Startup
  Success/Fail Dataset from Crunchbase*.
- **965 Israeli records removed:** the current file contains no row with
  `country_code = "ISR"`. The already-filtered files were committed in `91a6a04`
  (`data/processed/`) and `bd8a0cb` (`data/raw/`); the removal itself is not
  visible in Git because the pre-filter file was never committed. The pre-filter
  file used by TAS in notebook 02 (executed 2026-10-31) had 66,368 rows; the
  committed file has 65,403, and the four status counts differ by exactly 965
  (operating -760, closed -77, acquired -86, ipo -42). **This removal is awaiting
  confirmation** from TAS / the repository owner and is not yet documented in the
  project docs.
- **Raw and processed copies are identical:** `data/processed/startup_data_no_isr.csv`
  is byte-for-byte the same file as `data/raw/startup_data.csv` (checked below),
  and no code reads the processed copy.
- **Documentation is out of date:** `docs/handoff_preprocessing.md` and
  `docs/methodology.md` still quote 66,368 raw rows and 13,334 target rows. The
  figures verified here are **65,403** and **13,129**.

All reported figures are computed at execution time; the four expected counts are
hard-coded on purpose as assertions.


## 0. Setup

The notebook lives in `notebooks/`, while the modules live in `src/` and the data
in `data/`. The cell below locates the repository root, imports the **official
team modules** (never re-implementing them), and sets display options.

Only `data_loader`, `features` and `targets` are imported at this stage:
`preprocessing` is deliberately excluded because this step must not transform
anything.


In [1]:
from pathlib import Path
import hashlib
import subprocess
import sys

import pandas as pd
from IPython.display import display

# The notebook lives in notebooks/; src/ and data/ live at the repository root.
ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import data_loader
import features
import targets

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 120)

DATA_PATH = ROOT / "data" / "raw" / "startup_data.csv"
PROCESSED_PATH = ROOT / "data" / "processed" / "startup_data_no_isr.csv"

print("Repository root :", ROOT)
print("Dataset         :", DATA_PATH.relative_to(ROOT), f"({DATA_PATH.stat().st_size / 1e6:.1f} MB)")
print("data_loader     :", Path(data_loader.__file__).relative_to(ROOT))
print("features        :", Path(features.__file__).relative_to(ROOT))
print("targets         :", Path(targets.__file__).relative_to(ROOT))

Repository root : F:\Project FDD
Dataset         : data\raw\startup_data.csv (10.8 MB)
data_loader     : src\data_loader.py
features        : src\features.py
targets         : src\targets.py


### 0.1 Provenance evidence

Two facts are checked rather than asserted in prose: the commit the notebook is
executed on, and whether the "processed" copy really is a duplicate of the raw one.


In [2]:
head = subprocess.run(
    ["git", "rev-parse", "--short", "HEAD"],
    cwd=ROOT,
    capture_output=True,
    text=True,
    check=True,
).stdout.strip()

sha_raw = hashlib.sha1(DATA_PATH.read_bytes()).hexdigest()
sha_processed = hashlib.sha1(PROCESSED_PATH.read_bytes()).hexdigest()

print(f"Current HEAD           : {head}")
print(f"Documented base commit : bd8a0cb")
print(f"SHA-1 data/raw/        : {sha_raw}")
print(f"SHA-1 data/processed/  : {sha_processed}")
print(f"Raw and processed are byte-identical: {sha_raw == sha_processed}")

Current HEAD           : bd8a0cb
Documented base commit : bd8a0cb
SHA-1 data/raw/        : 1d425a91c0cf1f2513c243ce426dd3ffa5f07932
SHA-1 data/processed/  : 1d425a91c0cf1f2513c243ce426dd3ffa5f07932
Raw and processed are byte-identical: True


## 1. Loading through the official chain

The dataset is built by calling the team functions in the order defined in
`docs/handoff_preprocessing.md`:

1. `data_loader.load_raw()` - reads the CSV, strips column names, filters nothing.
2. `features.build_features(max_year=2026)` - cleans the four geographic columns
   (`country_code`, `state_code`, `region`, `city`), derives `primary_category`
   and `founding_year`, and adds `founding_month`, which is EDA-only. Together
   these are the six protocol features.
3. `targets.prepare_target()` - keeps `ipo`, `acquired`, `closed`, drops
   `operating`, and creates the binary `target`.

Nothing is imputed, encoded or scaled here.


In [3]:
raw = data_loader.load_raw(DATA_PATH)
with_features = features.build_features(raw, max_year=2026)
dataset = targets.prepare_target(with_features)

print(f"load_raw        : {raw.shape[0]:>6,} rows x {raw.shape[1]:>2} columns")
print(f"build_features  : {with_features.shape[0]:>6,} rows x {with_features.shape[1]:>2} columns")
print(f"prepare_target  : {dataset.shape[0]:>6,} rows x {dataset.shape[1]:>2} columns")

load_raw        : 65,403 rows x 14 columns
build_features  : 65,403 rows x 17 columns
prepare_target  : 13,129 rows x 18 columns


### 1.1 Verification of the expected counts

These four numbers are the contract of this deliverable. If any of them changes,
the notebook **stops with an explicit error** instead of continuing with a
different dataset: that would mean the data or the team code changed, and the
figures in the report would no longer be comparable.


In [4]:
EXPECTED_RAW_ROWS = 65_403
EXPECTED_TARGET_ROWS = 13_129
EXPECTED_SUCCESS = 6_968
EXPECTED_FAILURE = 6_161

target_counts = dataset["target"].value_counts().sort_index()
success = int(target_counts.get(1, 0))
failure = int(target_counts.get(0, 0))

assert raw.shape[0] == EXPECTED_RAW_ROWS, (
    f"Raw row count changed: {raw.shape[0]:,} != {EXPECTED_RAW_ROWS:,}"
)
assert dataset.shape[0] == EXPECTED_TARGET_ROWS, (
    f"Target-prepared row count changed: {dataset.shape[0]:,} != {EXPECTED_TARGET_ROWS:,}"
)
assert success == EXPECTED_SUCCESS, (
    f"Success count changed: {success:,} != {EXPECTED_SUCCESS:,}"
)
assert failure == EXPECTED_FAILURE, (
    f"Failure count changed: {failure:,} != {EXPECTED_FAILURE:,}"
)

verification = pd.DataFrame(
    [
        ("raw rows", raw.shape[0], EXPECTED_RAW_ROWS),
        ("target-prepared rows", dataset.shape[0], EXPECTED_TARGET_ROWS),
        ("success (target = 1)", success, EXPECTED_SUCCESS),
        ("failure (target = 0)", failure, EXPECTED_FAILURE),
    ],
    columns=["check", "observed", "expected"],
)
verification["match"] = verification["observed"] == verification["expected"]
display(verification)
print("All four expected counts match.")

,check,observed,expected,match
0,raw rows,65403,65403,True
1,target-prepared rows,13129,13129,True
2,success (target = 1),6968,6968,True
3,failure (target = 0),6161,6161,True


All four expected counts match.


## 2. Dataset dimensions

Row and column counts at each stage of the chain. The row count only changes at
`prepare_target` (exclusion of `operating`); the column count grows by three
derived columns (`primary_category`, `founding_year`, `founding_month`) and
`target` (14 -> 17 -> 18).


In [5]:
dimensions = pd.DataFrame(
    [
        ("raw (data_loader.load_raw)", *raw.shape),
        ("+ features (features.build_features)", *with_features.shape),
        ("+ target (targets.prepare_target)", *dataset.shape),
    ],
    columns=["stage", "rows", "columns"],
)
display(dimensions)

,stage,rows,columns
0,raw (data_loader.load_raw),65403,14
1,+ features (features.build_features),65403,17
2,+ target (targets.prepare_target),13129,18


## 3. Columns and data types

The raw file has 14 columns, all stored as text except `funding_rounds`. The
chain adds derived columns; their types matter because `founding_year` is a
`float64` (so that scikit-learn can impute it later) while the categorical
features stay nullable text.

`funding_total_usd` is text rather than numeric because Crunchbase uses `"-"` as
a missing marker.


In [6]:
raw_dtypes = raw.dtypes.astype(str).to_frame("dtype")
raw_dtypes.index.name = "column"
print(f"Raw columns ({raw.shape[1]}):")
display(raw_dtypes)

added_features = [column for column in with_features.columns if column not in raw.columns]
added_target = [column for column in dataset.columns if column not in with_features.columns]
print("Added by features.build_features :", added_features)
print("Added by targets.prepare_target :", added_target)

final_dtypes = dataset.dtypes.astype(str).to_frame("dtype")
final_dtypes.index.name = "column"
print(f"\nTarget-prepared columns ({dataset.shape[1]}):")
display(final_dtypes)

Raw columns (14):


,dtype
column,
permalink,object
name,object
homepage_url,object
category_list,object
funding_total_usd,object
status,object
country_code,object
state_code,object
region,object


Added by features.build_features : ['primary_category', 'founding_year', 'founding_month']
Added by targets.prepare_target : ['target']

Target-prepared columns (18):


,dtype
column,
permalink,object
name,object
homepage_url,object
category_list,object
funding_total_usd,object
status,string
country_code,string
state_code,string
region,string


## 4. Target distribution

`target = 1` groups IPO and acquisition (success under the project convention),
`target = 0` is closure. `operating` is excluded, which is why the target-prepared
set is much smaller than the raw file.

The class balance is close to 50/50, so no resampling is required at this stage.


In [7]:
target_distribution = pd.DataFrame(
    {
        "rows": target_counts,
        "share (%)": (100 * target_counts / len(dataset)).round(2),
    }
)
target_distribution.index = ["0 - failure (closed)", "1 - success (ipo / acquired)"]
display(target_distribution)

raw_status = raw["status"].value_counts().to_frame("rows")
raw_status["share (%)"] = (100 * raw_status["rows"] / len(raw)).round(1)
print("Raw status distribution (before exclusion of operating):")
display(raw_status)

print("Status vs target in the target-prepared dataset:")
display(pd.crosstab(dataset["status"], dataset["target"], margins=True, margins_name="total"))

,rows,share (%)
0 - failure (closed),6161,46.93
1 - success (ipo / acquired),6968,53.07


Raw status distribution (before exclusion of operating):


,rows,share (%)
status,,
operating,52274,79.9
closed,6161,9.4
acquired,5463,8.4
ipo,1505,2.3


Status vs target in the target-prepared dataset:


target,0,1,total
status,,,
acquired,0,5463,5463
closed,6161,0,6161
ipo,0,1505,1505
total,6161,6968,13129


## 5. Missing values

Counts and percentages are reported on the raw file and on the target-prepared
file. **No value is filled here**: imputation belongs to
`src/preprocessing.py` pipelines, learned on the training set only.

The last table restricts the view to the six protocol features, because those are
the columns the models will actually consume.


In [8]:
def missing_table(frame: pd.DataFrame) -> pd.DataFrame:
    counts = frame.isna().sum()
    table = pd.DataFrame(
        {
            "column": counts.index,
            "missing": counts.to_numpy(),
            "share (%)": (100 * counts.to_numpy() / len(frame)).round(1),
        }
    )
    table = table[table["missing"] > 0].sort_values("missing", ascending=False)
    return table.reset_index(drop=True)

print(f"Raw dataset - {len(raw):,} rows")
display(missing_table(raw))

print(f"Target-prepared dataset - {len(dataset):,} rows")
display(missing_table(dataset))

protocol_missing = pd.DataFrame(
    {
        "column": list(features.FEATURE_COLUMNS),
        "missing": [int(dataset[column].isna().sum()) for column in features.FEATURE_COLUMNS],
    }
)
protocol_missing["share (%)"] = (100 * protocol_missing["missing"] / len(dataset)).round(1)
print("Missing values among the six protocol features:")
display(protocol_missing)

Raw dataset - 65,403 rows


,column,missing,share (%)
0,founded_at,15089,23.1
1,state_code,8447,12.9
2,region,7928,12.1
3,city,7926,12.1
4,country_code,6958,10.6
5,homepage_url,5018,7.7
6,category_list,3126,4.8
7,first_funding_at,23,0.0
8,name,1,0.0


Target-prepared dataset - 13,129 rows


,column,missing,share (%)
0,founding_month,3719,28.3
1,founding_year,3719,28.3
2,founded_at,3698,28.2
3,state_code,2190,16.7
4,city,2133,16.2
5,region,2133,16.2
6,country_code,1991,15.2
7,homepage_url,1209,9.2
8,primary_category,1077,8.2
9,category_list,1077,8.2


Missing values among the six protocol features:


,column,missing,share (%)
0,country_code,1991,15.2
1,state_code,2190,16.7
2,region,2133,16.2
3,city,2133,16.2
4,primary_category,1077,8.2
5,founding_year,3719,28.3


## 6. Duplicate checks

Three levels are checked: fully duplicated rows, duplicated `permalink` (the
Crunchbase identifier) and duplicated `name`.

A duplicated name is **not** necessarily an error: different companies can share a
name. It is reported for information, not as a defect to fix in this step.


In [9]:
duplicates = pd.DataFrame(
    [
        (
            "exact duplicate rows",
            int(raw.duplicated().sum()),
            int(dataset.duplicated().sum()),
        ),
        (
            "duplicate permalink",
            int(raw["permalink"].duplicated().sum()),
            int(dataset["permalink"].duplicated().sum()),
        ),
        (
            "duplicate name",
            int(raw["name"].duplicated().sum()),
            int(dataset["name"].duplicated().sum()),
        ),
    ],
    columns=["check", "raw", "target-prepared"],
)
display(duplicates)

,check,raw,target-prepared
0,exact duplicate rows,0,0
1,duplicate permalink,0,0
2,duplicate name,262,8


## 7. Founding year: range and missing values

`founding_year` comes from `features.split_founded_at`, which masks dates that
cannot be parsed or fall outside pandas' timestamp range, as well as implausible
years (before 1900 or after `max_year=2026`). `founded_at` itself keeps the
original text, so the two columns do not necessarily report the same number of
missing values: a non-null `founded_at` can still yield a missing `founding_year`
when its date is rejected.

This matters for the Mann-Whitney test planned in the next step: more than a
quarter of the target-prepared rows have no usable founding year.


In [10]:
founding_year = dataset["founding_year"]
founded_dates = pd.to_datetime(dataset["founded_at"], errors="coerce", format="mixed")
raw_years = pd.to_datetime(raw["founded_at"], errors="coerce", format="mixed").dt.year

founding_summary = pd.DataFrame(
    [
        ("rows", f"{len(dataset):,}"),
        ("founding_year present", f"{int(founding_year.notna().sum()):,}"),
        ("founding_year missing", f"{int(founding_year.isna().sum()):,}"),
        ("missing share (%)", f"{100 * founding_year.isna().mean():.1f}"),
        ("min", f"{founding_year.min():.0f}"),
        ("max", f"{founding_year.max():.0f}"),
        ("median", f"{founding_year.median():.0f}"),
    ],
    columns=["metric", "value"],
)
display(founding_summary)

# Decades are built from founding_year (plausibility mask applied), not from the
# raw text, so implausible dates are not counted twice.
decades = (founding_year // 10 * 10).value_counts().sort_index()
decades.index = decades.index.astype(int)
print("Rows with a usable founding_year, per decade:")
display(decades.rename_axis("decade").to_frame("rows"))

print("Raw founded_at missing                         :", f"{int(raw['founded_at'].isna().sum()):,}")
print("Raw founded_at before 1900 (masked by features):", f"{int((raw_years < 1900).sum()):,}")
print("Raw founded_at after 2026  (masked by features):", f"{int((raw_years > 2026).sum()):,}")
print("Raw founded_at out-of-range/unparseable (masked):", f"{int(raw_years.isna().sum() - raw['founded_at'].isna().sum()):,}")
print("Labeled founded_at before 1900 (masked)        :", f"{int((founded_dates.dt.year < 1900).sum()):,}")
print("Labeled founded_at after 2026  (masked)        :", f"{int((founded_dates.dt.year > 2026).sum()):,}")
print("Labeled founded_at out-of-range/unparseable    :", f"{int(founded_dates.isna().sum() - dataset['founded_at'].isna().sum()):,}")

,metric,value
0,rows,"13,129"
1,founding_year present,"9,410"
2,founding_year missing,"3,719"
3,missing share (%),28.3
4,min,1901
5,max,2016
6,median,2007


Rows with a usable founding_year, per decade:


,rows
decade,
1900,7
1910,5
1920,7
1930,8
1940,11
1950,12
1960,36
1970,75
1980,297


Raw founded_at missing                         : 15,089
Raw founded_at before 1900 (masked by features): 105
Raw founded_at after 2026  (masked by features): 3
Raw founded_at out-of-range/unparseable (masked): 4
Labeled founded_at before 1900 (masked)        : 18
Labeled founded_at after 2026  (masked)        : 1
Labeled founded_at out-of-range/unparseable    : 2


## 8. Primary-category cardinality

`primary_category` is the first entry of `category_list`. It is the variable used
by the planned chi-square test, so its cardinality decides whether that test is
valid as-is or needs a grouping of rare categories.

The chi-square test itself is **not** run in this step.


In [11]:
category = dataset["primary_category"]
category_counts = category.value_counts()

print(f"distinct primary_category values : {category.nunique():,}")
print(f"missing primary_category         : {int(category.isna().sum()):,} ({100 * category.isna().mean():.1f} %)")
print(f"categories with fewer than 30 rows: {int((category_counts < 30).sum()):,} of {len(category_counts):,}")

top_categories = category_counts.head(10).to_frame("rows")
top_categories["share (%)"] = (100 * top_categories["rows"] / len(dataset)).round(1)
display(top_categories)

coverage = 100 * category_counts.head(10).sum() / len(dataset)
print(f"Rows covered by the top 10 categories: {coverage:.1f} %")

distinct primary_category values : 503
missing primary_category         : 1,077 (8.2 %)
categories with fewer than 30 rows: 438 of 503


,rows,share (%)
primary_category,,
Biotechnology,1079,8.2
Software,946,7.2
Curated Web,741,5.6
Advertising,582,4.4
E-Commerce,437,3.3
Mobile,409,3.1
Enterprise Software,364,2.8
Games,306,2.3
Analytics,281,2.1


Rows covered by the top 10 categories: 41.2 %


## 9. Baseline summary

Consolidated view of the facts established above. Every value is computed from the
data loaded in this run; nothing is copied from the project documentation.

**Known limitations to keep in mind for the next steps**

- `founding_year` is missing for more than a quarter of the target-prepared rows.
- `founding_year` is also the main confounder of the target: older startups have
  had more time to be acquired, listed or closed. The Mann-Whitney test will
  describe this association, not a causal effect.
- `primary_category` has several hundred levels, many of them tiny: the chi-square
  test will need an explicit grouping rule.
- The 965 removed Israeli rows are not documented in the project documentation;
  all figures here are conditional on that filter.


In [12]:
baseline = pd.DataFrame(
    [
        ("raw rows", f"{len(raw):,}"),
        ("raw columns", f"{raw.shape[1]}"),
        ("target-prepared rows", f"{len(dataset):,}"),
        (
            "success (target = 1)",
            f"{success:,} ({100 * success / len(dataset):.2f} %)",
        ),
        (
            "failure (target = 0)",
            f"{failure:,} ({100 * failure / len(dataset):.2f} %)",
        ),
        ("exact duplicate rows", f"{int(dataset.duplicated().sum())}"),
        ("duplicate permalink", f"{int(dataset['permalink'].duplicated().sum())}"),
        (
            "founding_year range",
            f"{founding_year.min():.0f} - {founding_year.max():.0f}",
        ),
        (
            "founding_year missing",
            f"{int(founding_year.isna().sum()):,} ({100 * founding_year.isna().mean():.1f} %)",
        ),
        ("distinct primary_category", f"{category.nunique():,}"),
        (
            "primary_category missing",
            f"{int(category.isna().sum()):,} ({100 * category.isna().mean():.1f} %)",
        ),
    ],
    columns=["baseline fact", "value"],
)
display(baseline)

,baseline fact,value
0,raw rows,"65,403"
1,raw columns,14
2,target-prepared rows,"13,129"
3,success (target = 1),"6,968 (53.07 %)"
4,failure (target = 0),"6,161 (46.93 %)"
5,exact duplicate rows,0
6,duplicate permalink,0
7,founding_year range,1901 - 2016
8,founding_year missing,"3,719 (28.3 %)"
9,distinct primary_category,503


## 10. Next step (not part of this deliverable)

1. Chi-square test of independence: `primary_category` (after grouping rare levels)
   vs `target`, with expected-count checks.
2. Mann-Whitney U test: `founding_year` in class 1 vs class 0, with the
   observation-window caveat.
3. Only then, the descriptive figures for the report.

The Streamlit dashboard (`src/dashboard_eda.py`) already covers descriptives,
missingness, histograms, frequencies, correlations and filters; this notebook
should not duplicate it. It should deliver the analysis and the hypothesis tests.
